# xLSTM, sLSTM, mLSTMs - with Checkpoints

## Preparation

### Import modules

In [1]:
# Cell 1: Mount & Navigate
from google.colab import drive
from torch.version import cuda

drive.mount('/content/drive')

# Go to correct folder
%cd /content/drive/MyDrive/Colab\ Notebooks/thesis/LSTM_Train

# Verify structure
!ls -la ../
# Should show: dataset/  thesis_utils/  LSTM_Train/

!pip install loguru torchxlstm fastparquet

import sys
from pathlib import Path

# Add thesis_utils to path (parent dir)
sys.path.insert(0, '/content/drive/MyDrive/Colab\ Notebooks/thesis')

# Or simpler:
sys.path.insert(0, str(Path.cwd().parent))

<>:20: SyntaxWarning: invalid escape sequence '\ '
<>:20: SyntaxWarning: invalid escape sequence '\ '
/tmp/ipython-input-2272185481.py:20: SyntaxWarning: invalid escape sequence '\ '
  sys.path.insert(0, '/content/drive/MyDrive/Colab\ Notebooks/thesis')


Mounted at /content/drive
/content/drive/MyDrive/Colab Notebooks/thesis/LSTM_Train
total 29
drwx------ 2 root root 4096 Dec 28 11:56  checkpoints
drwx------ 2 root root 4096 Dec 25 14:57  dataset
drwx------ 2 root root 4096 Dec 25 18:09  GRU_Train
drwx------ 2 root root 4096 Dec 25 14:56  LSTM_Train
drwx------ 2 root root 4096 Dec 26 10:59 'MLSTM   SLSTM'
-rw------- 1 root root  662 Dec 25 15:12  requirements_colab.txt
drwx------ 2 root root 4096 Dec 25 14:56  thesis_utils
drwx------ 2 root root 4096 Dec 25 22:13  XLSTM_Train
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 83.0 MB/s eta 0:00:00


In [2]:
# Prediction using LSTM, GRU-LSTM, xLSTM
import copy
import math
import os
from typing import List

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn.utils import clip_grad_norm_
from torch.optim import Optimizer
from torch.optim.lr_scheduler import LRScheduler
from torch.utils.data import DataLoader, Dataset, Subset

from sklearn.model_selection import KFold, GroupShuffleSplit

import thesis_utils as tu

from torch.amp import GradScaler, autocast

In [3]:
CHKPT_DIR = "/content/drive/MyDrive/Colab Notebooks/thesis/checkpoints"
os.makedirs(CHKPT_DIR, exist_ok=True)

In [4]:
def ckpt_path(serial, fold):
  return os.path.join(CHKPT_DIR, f"{serial}_fold{fold}.pt")

In [5]:
def safe_load_ckpt(path: str, map_location="cpu"):
  """Load a torch checkpoint, but delete it and return None if corrupted."""
  if not os.path.exists(path):
    return None

  try:
    return torch.load(path, map_location=map_location)
  except Exception as e:
    print(f"⚠️ Corrupted checkpoint detected: {path}")
    print(f"   Error: {e}")
    # Delete immediately so it cannot be loaded again later
    try:
      os.remove(path)
      print("   Deleted corrupted checkpoint.")
    except Exception as del_e:
      print(f"   Could not delete checkpoint: {del_e}")
    return None

### Configuration

In [6]:
# Model parameters
HORIZON = 1
BATCH_SIZE = 2048  # Increased to saturate GPU
EMBEDDING_SIZE = 256
NUM_EPOCHS = 100
HIDDEN_SIZE = 768
N_LAYERS = 3
DROPOUT = 0.05
XLSTM_TYPE = "M"
N_LAGS = 5

# Train parameters
TARGET = "EXPORT_centered"
FEATURES = [
  "contig", "comlang_off", "colony", "smctry",
]
N_SPLITS = 8
PATIENCE = 10
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-3
RANDOM_SEED = 16
SUBSAMPLE_ENABLED = False
N_DYADS = 10
XLSTM_LAYERS = "sm"

SANCTION_COLS = ["arms", "military", "trade", "travel", "other", "financial"]

# Torch config
torch.manual_seed(RANDOM_SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ----------------------------
# Colab / L4 performance knobs
# ----------------------------
if device.type == "cuda":
  print("GPU:", torch.cuda.get_device_name(0))
  print("CUDA:", torch.version.cuda, "| PyTorch:", torch.__version__)
  !nvidia-smi -L

  # cuDNN autotuner (best when shapes are stable, typical in training)
  torch.backends.cudnn.benchmark = True

  # Better GEMM kernels on Ampere+ (L4 is Ada)
  torch.set_float32_matmul_precision("high")

  # Reduce allocator fragmentation for long runs
  os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

# Mixed precision (L4 supports bf16 well)
USE_AMP = True
AMP_DTYPE = torch.bfloat16 if (
    device.type == "cuda" and getattr(torch.cuda, "is_bf16_supported", lambda: False)()) else torch.float16

# torch.compile can improve throughput (PyTorch 2.0+)
USE_TORCH_COMPILE = True
COMPILE_MODE = "max-autotune"  # alternatives: "reduce-overhead", "default"

# GradScaler is needed for fp16; for bf16 it should be disabled
USE_GRAD_SCALER = (device.type == "cuda" and USE_AMP and AMP_DTYPE == torch.float16)

print("Using device:", device)
if torch.cuda.is_available():
  print(torch.cuda.get_device_name(0))
  # Enable TF32 for faster computing on Ampere+ GPUs
  torch.backends.cuda.matmul.allow_tf32 = True
  torch.backends.cudnn.allow_tf32 = True

dyads_case_study = [
  # "USA_CHN", "CHN_USA",
  # "USA_CAN", "CAN_USA",
  # "DEU_CHN", "CHN_DEU",
  # "USA_DEU", "DEU_USA",
  # "USA_MEX", "MEX_USA",
  # "AUS_CHN", "CHN_AUS",
  # "USA_JPN", "JPN_USA",
  # "DEU_JPN", "JPN_DEU",
  # "USA_AUS", "AUS_USA",
  # "DEU_RUS", "RUS_DEU",
]

GPU: NVIDIA L4
CUDA: 12.6 | PyTorch: 2.9.0+cu126
GPU 0: NVIDIA L4 (UUID: GPU-a00aafde-b09e-2732-8be5-3f32c5036c22)
Using device: cuda
NVIDIA L4


/usr/local/lib/python3.12/dist-packages/torch/__init__.py:1617: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  _C._set_float32_matmul_precision(precision)


In [7]:
# Save config
SAVE_ENABLED = False
layers_string = f"({XLSTM_LAYERS})"
SERIAL_NUMBER = (
  f"{XLSTM_TYPE}LSTM"
  f"{layers_string if XLSTM_TYPE == 'X' else ''}"
  f"-{LEARNING_RATE}lr-{DROPOUT}d-{HIDDEN_SIZE}hs-{WEIGHT_DECAY}wd-{BATCH_SIZE}bs-{N_LAYERS}layers-{EMBEDDING_SIZE}es-kfolds{N_SPLITS}-hp"
)
SERIAL_NUMBER = SERIAL_NUMBER.replace(".", "_")
PATH_TO_FOLDER = ""

### Load Data

In [8]:
from pandas import DataFrame

processed = pd.read_parquet(path="../dataset/processed.parquet", engine="fastparquet")
df: DataFrame = processed.copy(deep=True)

### Sort, shift and compute data

In [9]:
# Sort data by Report + Partner + Year
df["dyad_id"] = df["ISO3_reporter"] + "_" + df["ISO3_partner"]
df = df.sort_values(by=["dyad_id", "Year"], ignore_index=True)

In [10]:
# Remove case study dyad_pairs
mask_keep = ~np.isin(df["dyad_id"], dyads_case_study)
df = df.loc[mask_keep].reset_index(drop=True)

In [11]:
# Sanity check case study pairs
has_overlap = df["dyad_id"].isin(dyads_case_study).any()

if has_overlap:
  print("⚠️ Some case study dyads are present in the DataFrame.")
else:
  print("✅ No case study dyads found in the DataFrame.")

✅ No case study dyads found in the DataFrame.


In [12]:
if SUBSAMPLE_ENABLED:
  dyad_subsample = pd.Series(df["dyad_id"].unique()).sample(n=N_DYADS, random_state=RANDOM_SEED, replace=False)
  df = df[df["dyad_id"].isin(dyad_subsample)]

print(f"Unique dyads: {df["dyad_id"].nunique()}")

Unique dyads: 33672


In [13]:
df["sanction"] = (df[SANCTION_COLS]
                  .sum(axis=1)).astype(int)

### Coerce numerical values and convert dyad_id to categorical

In [14]:
num_cols = ["distw", "GDP_reporter", "GDP_partner", "sanction", "contig",
            "comlang_off", "colony", "smctry", "Year", ]
df[num_cols] = df[num_cols].apply(pd.to_numeric, errors="coerce").astype(float)
df = df.dropna(subset=num_cols)

In [15]:
df["Year"] = df["Year"].astype(int)
for col in ["dyad_id"]:
  df[col] = pd.Categorical(df[col], categories=sorted(df[col].unique()))

In [16]:
# Save EXPORT std and median to undo centering
EXPORT_STD = df["EXPORT"].std()
EXPORT_MEDIAN = df["EXPORT"].median()

### Center data

In [17]:
center_columns = ["distw", "GDP_reporter", "GDP_partner", "EXPORT"]
for col in center_columns:
  median = df[col].median()
  std_df = df[col].std()
  df[col + "_centered"] = (df[col] - median) / std_df
FEATURES += ["distw_centered"]

In [18]:
lag_cols = ["GDP_reporter_centered", "GDP_partner_centered", "sanction"]
for col in lag_cols:
  for index in range(1, N_LAGS + 1):
    df[f"{col}_lag{index}"] = df.groupby("dyad_id", observed=True)[col].shift(index)

In [19]:
df = df.dropna()

In [20]:
FEATURES += [f"{c}_lag{index}" for c in lag_cols for index in range(1, N_LAGS + 1)]

## Split data

In [21]:
# Embeddings
dyad_to_idx = { dyad: i for i, dyad in enumerate(df["dyad_id"].cat.categories) }
df["dyad_idx"] = df["dyad_id"].map(dyad_to_idx).astype(int)

In [22]:
# Split into Train, Validation and Test sets
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_SEED)

train_idx, test_idx = next(gss.split(df, groups=df["dyad_id"]))
test_df = df.iloc[test_idx]
train_df = df.iloc[train_idx]

train_idx, val_idx = next(gss.split(train_df, groups=train_df["dyad_id"]))
val_df = train_df.iloc[val_idx]
train_df = train_df.iloc[train_idx]

In [23]:
train_df.loc[:, FEATURES] = train_df.loc[:, FEATURES].astype(
  "float32",
  copy=False
)

# Train

## Define Fold and Epoch steps
_For reusability_

In [24]:
# Create KFold object
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_SEED)

In [25]:
def epoch_step(
    model: nn.Module,
    optimizer: Optimizer,
    criterion: nn.Module,
    scheduler: LRScheduler,
    train_loader: DataLoader,
    val_loader: DataLoader,
    device: any,
    scaler: torch.amp.GradScaler,
) -> float:
  # =========================
  # TRAIN
  # =========================
  model.train()

  for X, y, di in train_loader:
    X, y, di = map(lambda t: t.to(device, non_blocking=True), (X, y, di))

    optimizer.zero_grad(set_to_none=True)

    # --- AMP forward ---
    with autocast("cuda"):
      y_pred = model(X, di)

      if not torch.isfinite(y_pred).all():
        print("⚠️ NaN or Inf detected in y_pred — stopping here!")
        return float("inf")

      loss = criterion(y_pred, y)

    if not torch.isfinite(loss):
      print("⚠️ loss is NaN or Inf!")
      return float("inf")

    # --- AMP backward ---
    scaler.scale(loss).backward()

    # IMPORTANT: unscale before clipping
    scaler.unscale_(optimizer)
    clip_grad_norm_(model.parameters(), max_norm=1.0)

    scaler.step(optimizer)
    scaler.update()

    # OneCycleLR MUST step per batch
    scheduler.step()

  # =========================
  # VALIDATION
  # =========================
  model.eval()
  val_losses = []

  with torch.no_grad():
    for X, y, di in val_loader:
      X, y, di = map(lambda t: t.to(device, non_blocking=True), (X, y, di))
      with autocast("cuda"):
        preds = model(X, di)
        val_losses.append(criterion(preds, y).item())

  val_rmse = math.sqrt(sum(val_losses) / len(val_losses))
  return val_rmse

In [26]:
def _make_loader(subset, *, batch_size: int, shuffle: bool, n_workers: int) -> DataLoader:
  """Colab-friendly DataLoader with aggressive prefetch + pinned memory."""
  kwargs = dict(
    batch_size=batch_size,
    shuffle=shuffle,
    num_workers=n_workers,
    pin_memory=True,
    persistent_workers=(n_workers > 0),
  )

  # Only valid when num_workers > 0
  if n_workers > 0:
    kwargs["prefetch_factor"] = 4

  return DataLoader(subset, **kwargs)

In [27]:
# Define fold step
def fold_step(
    fold: int,
    train_idx: List,
    val_idx: List,
    dataset: Dataset,
    batch_size: int,
    num_epochs: int,
    model: nn.Module,
    device: any,
    optimizer: Optimizer,
    criterion: nn.Module,
    serial: str,
):
  # Use all available cores
  n_workers = min(8, (os.cpu_count() or 2))

  train_loader = _make_loader(Subset(dataset, train_idx), batch_size=batch_size, shuffle=True, n_workers=n_workers)
  val_loader = _make_loader(Subset(dataset, val_idx), batch_size=batch_size, shuffle=False, n_workers=n_workers)

  scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=7e-4,
    epochs=NUM_EPOCHS,
    steps_per_epoch=len(train_loader),
    div_factor=25,
    final_div_factor=100,
    pct_start=0.3,
    anneal_strategy="cos",
    three_phase=False,
  )

  ckpt_file = ckpt_path(serial, fold)
  tmp_ckpt_file = ckpt_file + ".tmp"

  ckpt = safe_load_ckpt(ckpt_file, map_location=device)

  start_epoch = 0
  best_rmse = float("inf")
  best_state = copy.deepcopy(model.state_dict())
  scaler = GradScaler("cuda", enabled=USE_GRAD_SCALER)

  if ckpt is not None:
    print(f"🔄 Resuming from checkpoint: {ckpt_file}")

    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    scheduler.load_state_dict(ckpt["scheduler_state"])
    scaler.load_state_dict(ckpt["scaler_state"])

    start_epoch = ckpt["epoch"] + 1
    best_rmse = ckpt["best_rmse"]
    best_state = ckpt["best_state"]

  print(f"Start epoch train for fold {fold}")

  for epoch in range(start_epoch, num_epochs):
    val_rmse = epoch_step(
      model=model,
      optimizer=optimizer,
      criterion=criterion,
      scheduler=scheduler,
      train_loader=train_loader,
      val_loader=val_loader,
      device=device,
      scaler=scaler,
    )

    print(f"Epoch {epoch + 1:02d}/{num_epochs} | val RMSE: {val_rmse:.4f}")

    if val_rmse < best_rmse - 1e-4:
      best_rmse = val_rmse
      best_state = copy.deepcopy(model.state_dict())

    # =========================
    # ATOMIC CHECKPOINT SAVE
    # =========================
    state = {
      "epoch": epoch,
      "model_state": model.state_dict(),
      "optimizer_state": optimizer.state_dict(),
      "scheduler_state": scheduler.state_dict(),
      "scaler_state": scaler.state_dict(),
      "best_rmse": best_rmse,
      "best_state": best_state,
    }

    torch.save(state, tmp_ckpt_file)
    os.replace(tmp_ckpt_file, ckpt_file)

  # =========================
  # FINAL EVALUATION
  # =========================
  model.load_state_dict(best_state)
  model.eval()

  preds_centered, truth_centered = [], []

  with torch.inference_mode():
    for X, y, di in val_loader:
      X, di = map(lambda t: t.to(device, non_blocking=True), (X, di))

      with autocast("cuda", enabled=(getattr(device, "type", "cpu") == "cuda" and USE_AMP)):
        out = model(X, di)

        preds_centered.append(out.float().cpu())
        truth_centered.append(y)

  preds_centered = torch.cat(preds_centered).numpy()
  truth_centered = torch.cat(truth_centered).numpy()

  # Convert back to raw scale
  preds_raw = preds_centered * EXPORT_STD + EXPORT_MEDIAN
  truth_raw = truth_centered * EXPORT_STD + EXPORT_MEDIAN

  rmse_centered = tu.rmse(truth_centered, preds_centered)
  mae_centered = tu.mae(truth_centered, preds_centered)
  rmae_centered = tu.rmae(truth_centered, preds_centered)
  pseudo_r2_centered = tu.pseudo_r2(truth_centered, preds_centered)

  print(
    f"Fold {fold} CENTERED  RMSE {rmse_centered:.4f} | "
    f"MAE {mae_centered:.4f} | R² {pseudo_r2_centered:.4f} | "
    f"RMAE {rmae_centered:.4f}"
  )

  rmse_raw = tu.rmse(truth_raw, preds_raw)
  mae_raw = tu.mae(truth_raw, preds_raw)
  rmae_raw = tu.rmae(truth_raw, preds_raw)
  pseudo_r2_raw = tu.pseudo_r2(truth_raw, preds_raw)

  print(
    f"Fold {fold} RAW  RMSE {rmse_raw:.4f} | "
    f"MAE {mae_raw:.4f} | R² {pseudo_r2_raw:.4f} | "
    f"RMAE {rmae_raw:.4f}"
  )

  return (
    { "RMSE": rmse_centered, "MAE": mae_centered, "R2": pseudo_r2_centered, "RMAE": rmae_centered },
    { "RMSE": rmse_raw, "MAE": mae_raw, "R2": pseudo_r2_raw, "RMAE": rmae_raw },
    copy.deepcopy(best_state),
  )

## Train Raw dataset

### Split dataset

In [28]:
# Convert df_scaled to pytorch Tensor
dataset, dyad_to_idx = tu.make_panel_datasets_dyad(
  data=df,
  features=FEATURES,
  target=TARGET,
  horizon=HORIZON,
)

In [29]:
# Create DataLoaders for the 3 sets
n_workers = min(8, (os.cpu_count() or 2))

train_kwargs = dict(
  batch_size=BATCH_SIZE,
  shuffle=True,
  num_workers=n_workers,
  persistent_workers=(n_workers > 0),
  pin_memory=True,
)
val_kwargs = dict(
  batch_size=BATCH_SIZE,
  shuffle=False,
  num_workers=n_workers,
  persistent_workers=(n_workers > 0),
  pin_memory=True,
)
test_kwargs = dict(
  batch_size=BATCH_SIZE,
  shuffle=False,
  num_workers=n_workers,
  persistent_workers=(n_workers > 0),
  pin_memory=True,
)

if n_workers > 0:
  train_kwargs["prefetch_factor"] = 4
  val_kwargs["prefetch_factor"] = 4
  test_kwargs["prefetch_factor"] = 4

train_loader = DataLoader(Subset(dataset, train_idx), **train_kwargs)
val_loader = DataLoader(Subset(dataset, val_idx), **val_kwargs)
test_loader = DataLoader(Subset(dataset, test_idx), **test_kwargs)

### Train model

In [30]:
# Save best train iteration
best_fold_state = None
best_fold_rmse = float("inf")

metrics_per_fold = {
  "RMSE": [],
  "MAE": [],
  "R2": [],
  "RMAE": [],
}

metrics_per_fold_raw = {
  "RMSE": [],
  "MAE": [],
  "R2": [],
  "RMAE": [],
}

In [ ]:
for fold, (train_idx, val_idx) in enumerate(kf.split(np.arange(len(dataset))), 1):

  ckpt_file = ckpt_path(SERIAL_NUMBER, fold)
  ckpt = safe_load_ckpt(ckpt_file, map_location="cpu")

  if ckpt is not None and ckpt.get("epoch", -1) >= NUM_EPOCHS - 1:
    print(f"✅ Fold {fold} already completed — skipping.")
    continue

  print(f"=== FOLD {fold}/{N_SPLITS} ===")

  model = tu.DyadXLSTM(
    n_features=len(FEATURES),
    n_dyads=len(dyad_to_idx),
    embed_dim=EMBEDDING_SIZE,
    hidden_size=HIDDEN_SIZE,
    dropout=DROPOUT,
    horizon=HORIZON,
    type=XLSTM_TYPE,
    layers=XLSTM_LAYERS,
    n_layers=N_LAYERS,
  ).to(device=device)

  # JIT Compile the model for speedup (requires PyTorch 2.0+)
  if USE_TORCH_COMPILE and hasattr(torch, "compile") and device.type == "cuda":
    try:
      model = torch.compile(model, mode=COMPILE_MODE)
      print(f"Model compiled with torch.compile(mode={COMPILE_MODE!r})")
    except Exception as e:
      print(f"Could not compile model: {e}")

  criterion = nn.SmoothL1Loss(beta=0.5)

  adamw_kwargs = dict(lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.98), eps=1e-8)
  if device.type == "cuda":
    try:
      optimizer = optim.AdamW(model.parameters(), **adamw_kwargs, fused=True)
      print("Using fused AdamW")
    except TypeError:
      optimizer = optim.AdamW(model.parameters(), **adamw_kwargs)
  else:
    optimizer = optim.AdamW(model.parameters(), **adamw_kwargs)

  fold_metrics, fold_metrics_raw, best_state = fold_step(fold=fold,
                                                         train_idx=train_idx,
                                                         val_idx=val_idx,
                                                         dataset=dataset,
                                                         batch_size=BATCH_SIZE,
                                                         num_epochs=NUM_EPOCHS,
                                                         model=model,
                                                         device=device,
                                                         optimizer=optimizer,
                                                         criterion=criterion,
                                                         serial=SERIAL_NUMBER)

  if fold_metrics["RMSE"] < best_fold_rmse:
    best_fold_rmse = fold_metrics["RMSE"]
    best_fold_state = copy.deepcopy(best_state)

  for k, v in fold_metrics.items():
    metrics_per_fold[k].append(v)

  for k, v in fold_metrics_raw.items():
    metrics_per_fold_raw[k].append(v)

=== FOLD 1/8 ===
Model compiled with torch.compile(mode='max-autotune')
Using fused AdamW
Start epoch train for fold 1


W1228 14:01:25.670000 1443 torch/_inductor/utils.py:1558] [0/0] Not enough SMs to use max_autotune_gemm mode
Autotune Choices Stats:
{"num_choices": 2, "num_triton_choices": 0, "best_kernel": "bias_addmm", "best_time": 0.060416001826524734}
AUTOTUNE addmm(2048x768, 2048x552, 552x768)
strides: [0, 1], [552, 1], [1, 552]
dtypes: torch.float16, torch.float16, torch.float16
  bias_addmm 0.0604 ms 100.0% 
  addmm 0.0850 ms 71.1% 
SingleProcess AUTOTUNE benchmarking takes 0.4876 seconds and 0.0004 seconds precompiling for 2 choices
Autotune Choices Stats:
{"num_choices": 2, "num_triton_choices": 0, "best_kernel": "bias_addmm", "best_time": 0.04095999896526337}
AUTOTUNE addmm(2048x768, 2048x276, 276x768)
strides: [0, 1], [276, 1], [1, 276]
dtypes: torch.float16, torch.float16, torch.float16
  bias_addmm 0.0410 ms 100.0% 
  addmm 0.0604 ms 67.8% 
SingleProcess AUTOTUNE benchmarking takes 0.0958 seconds and 0.0003 seconds precompiling for 2 choices
Autotune Choices Stats:
{"num_choices": 2, "nu

Epoch 01/100 | val RMSE: 0.2488
Epoch 02/100 | val RMSE: 0.2215
Epoch 03/100 | val RMSE: 0.1889
Epoch 04/100 | val RMSE: 0.1760
Epoch 05/100 | val RMSE: 0.1700
Epoch 06/100 | val RMSE: 0.1669
Epoch 07/100 | val RMSE: 0.1626
Epoch 08/100 | val RMSE: 0.1572
Epoch 09/100 | val RMSE: 0.1600
Epoch 10/100 | val RMSE: 0.1524
Epoch 11/100 | val RMSE: 0.1560
Epoch 12/100 | val RMSE: 0.1529
Epoch 13/100 | val RMSE: 0.1594
Epoch 14/100 | val RMSE: 0.1713
Epoch 15/100 | val RMSE: 0.1706
Epoch 16/100 | val RMSE: 0.1975
Epoch 17/100 | val RMSE: 0.1953
Epoch 18/100 | val RMSE: 0.1686


## Save Model

In [ ]:
torch.save({
  "model_state_dict": best_fold_state,
  "model_hyperparams": {
    "n_features": len(FEATURES),
    "n_dyads": len(dyad_to_idx),
    "n_layers": N_LAYERS,
    "embed_dim": EMBEDDING_SIZE,
    "hidden_size": HIDDEN_SIZE,
    "dropout": DROPOUT,
    "horizon": HORIZON,
    "type": XLSTM_TYPE,
    "layers": XLSTM_LAYERS,
  },
  "dyad_to_idx": dyad_to_idx,
  "feature_names": FEATURES,
}, PATH_TO_FOLDER + SERIAL_NUMBER + ".pt")
print("Saved model to", PATH_TO_FOLDER + SERIAL_NUMBER + ".pt")

In [ ]:
def summarize(xs):
  xs = np.asarray(xs, dtype=float)
  n = xs.size
  mean = xs.mean()
  std = xs.std(ddof=1)  # sample std
  se = std / math.sqrt(n)
  try:
    from scipy.stats import t
    tcrit = t.ppf(0.975, df=n - 1)
  except Exception:
    tcrit = 1.96  # normal approx≈
  ci95 = tcrit * se
  return mean, std, ci95

In [ ]:
print("\n=== Cross-fold CENTERED summary ===")
for name in ["MAE", "RMSE", "R2", "RMAE"]:
  mean, std, ci = summarize(metrics_per_fold[name])
  print(f"{name:>5}: {mean:.4f} ± {std:.4f}  (95% CI ±{ci:.4f})")

print("\n=== Cross-fold RAW summary ===")
for name in ["MAE", "RMSE", "R2", "RMAE"]:
  mean, std, ci = summarize(metrics_per_fold_raw[name])
  print(f"{name:>5}: {mean:.4f} ± {std:.4f}  (95% CI ±{ci:.4f})")